# 03 · Detection Tuning **v2**

## v1'den öğrendiklerimiz
1. ❌ **Hipotez çürüdü:** footprint/sigma büyütmek recall'ı **düşürdü** (0.76→0.59).
   Tepe birleştirmek sahte değil **gerçek** hücreleri siliyor. Mevcut ayar en iyisiydi.
2. 🔍 **Eşik bağlayıcı değil:** `otsu`→`otsu*0.8` recall'ı hiç değiştirmedi (0.760→0.760).
   Kısıtlayan → **yerel-maksimum koşulu**.
3. 📊 **Yoğunluk dataset'e özgü:** 58/kare (`6bba_05b6850b`) vs 634/kare (`6bba_05db0fb1`).
   → Global "213 hedefi" **yanlış**; `T_true` dataset başına.
4. 🎯 **Kırık dataset:** `44b6_0b24845f` — GT parlaklık 1953 **<** hacim p99 2589
   (GT'den parlak yapılar var) → GT çekirdekleri yerel-maks olamıyor, en yakın tespit **6–10 µm**.
5. 💡 **Recall doymadı** (dens 45→315 ⇒ recall 0.48→0.76) ve ceza **zayıf** (2× → yalnız 0.9).
   `jaccard ≈ recall²` olduğuna göre **agresif tespit kârlı olabilir**.

## v2'de düzeltilenler
- **Havuzlanmış recall** (toplam eşleşen / toplam GT) — v1'de kare-başına ortalama alıyordum;
  44b6'da 1–2 GT/kare, 6bba'da 11–17 → eşit ağırlık **istatistiksel olarak çürüktü**.
  (Yarışma metriği de videolar arası **micro-average** → havuzlama doğru olan.)
- Grid **daha çok tespit** yönüne uzatıldı (küçük footprint, düşük eşik).
- Sahte global ceza yerine → en iyi konfigler **tam tracking + gerçek metrikle** doğrulanır.
- Kırık dataset'in kaçırılan GT'leri **görselleştirilir**.

## 0 · Kurulum

In [ ]:
import sys, subprocess, os, time, itertools
def _scan(base):
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if not d.endswith((".zarr",".geff")) and d!="competitions"]
        if root.count(os.sep) > 9: dirs[:]=[]; continue
        yield root, files
def ensure_zarr():
    try:
        import zarr; return zarr
    except ImportError: pass
    for root, files in _scan("/kaggle/input"):
        if os.path.basename(root)=="zarr" and "__init__.py" in files:
            p=os.path.dirname(root); sys.path.insert(0,p)
            try:
                import zarr; print("zarr <- sys.path:",p); return zarr
            except ImportError: sys.path.pop(0)
    subprocess.run([sys.executable,"-m","pip","install","-q","zarr"],check=False)
    import zarr; return zarr
zarr=ensure_zarr(); print("zarr:",zarr.__version__)

import numpy as np, pandas as pd
from pathlib import Path
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.filters import threshold_otsu
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

SCALE=(1.625,0.40625,0.40625); S=np.array(SCALE,dtype=np.float32)
MATCH_UM=7.0; LINK_MAX_UM=8.0
FIG=Path("/kaggle/working/figures"); FIG.mkdir(parents=True,exist_ok=True)

INPUT=Path("/kaggle/input")
def find_root():
    st=[(INPUT,0)]
    while st:
        b,d=st.pop()
        try:
            if (b/"train").is_dir() and (b/"test").is_dir(): return b
        except Exception: pass
        if d<4:
            for c in sorted(b.iterdir()):
                if c.is_dir() and not c.name.endswith((".zarr",".geff")): st.append((c,d+1))
ROOT=find_root(); TRAIN=ROOT/"train"; TEST=ROOT/"test"
test_names=sorted(p.stem for p in TEST.glob("*.zarr"))

def open_image(zp):
    n=zarr.open(str(zp),mode="r"); a=dict(n.attrs)
    ms=a.get("multiscales") or (a.get("ome") or {}).get("multiscales")
    if ms: return n[ms[0]["datasets"][0]["path"]]
    return n["0"] if "0" in list(n.keys()) else n
def load_geff(gp):
    g=zarr.open(str(gp),mode="r"); nodes=g["nodes"]
    ids=np.asarray(nodes["ids"]); props={}
    for pn in list(nodes["props"].keys()):
        try: props[pn]=np.asarray(nodes["props"][pn]["values"])
        except Exception: pass
    d={"id":ids}
    for k in ("t","z","y","x"):
        if k in props: d[k]=props[k]
    return pd.DataFrame(d), np.asarray(g["edges"]["ids"])
print("test:",test_names)

## 1 · Kareleri yükle — **GT sayısına göre adaptif**
Seyrek dataset'lerden (44b6: 1–2 GT/kare) daha çok kare alırız ki havuzlanmış recall
istatistiksel olarak anlamlı olsun.

In [ ]:
TARGET_GT=40; MAX_FR=12
FRAMES=[]; GT_ALL={}
t0=time.time()
for nm in test_names:
    gdf,ge=load_geff(TRAIN/(nm+".geff")); GT_ALL[nm]=(gdf,ge)
    arr=open_image(TEST/(nm+".zarr"))
    cnt=gdf.groupby("t").size().sort_values(ascending=False)
    picks=[]; tot=0
    for t,c in cnt.items():
        picks.append(int(t)); tot+=int(c)
        if tot>=TARGET_GT or len(picks)>=MAX_FR: break
    for t in picks:
        v=np.asarray(arr[t]).astype(np.float32)
        gp=gdf[gdf.t==t][["z","y","x"]].values.astype(np.float32)
        FRAMES.append((nm,t,v,gp))
print(f"{len(FRAMES)} kare, {time.time()-t0:.0f}s, RAM ~{sum(f[2].nbytes for f in FRAMES)/1e6:.0f} MB")
for nm in test_names:
    fr=[f for f in FRAMES if f[0]==nm]
    print(f"  {nm}: {len(fr):2d} kare, {sum(len(f[3]) for f in fr):3d} GT node")

## 2 · Detection + **havuzlanmış** değerlendirme

In [ ]:
def detect(v, sigma, foot, thr_mode):
    sm=ndi.gaussian_filter(v, sigma=sigma)
    if thr_mode=="otsu": thr=threshold_otsu(sm)
    elif thr_mode.startswith("otsu*"): thr=threshold_otsu(sm)*float(thr_mode.split("*")[1])
    elif thr_mode.startswith("p"): thr=np.percentile(sm, float(thr_mode[1:]))
    mx=ndi.maximum_filter(sm, size=foot)
    peaks=(sm==mx)&(sm>thr)
    lbl,n=ndi.label(peaks)
    if n==0: return np.zeros((0,3),np.float32)
    return np.asarray(ndi.center_of_mass(sm,lbl,np.arange(1,n+1)),dtype=np.float32)

def sweep_eval(sigma, foot, thr):
    # HAVUZLANMIS: toplam eslesen / toplam GT (kare-basi ortalama DEGIL)
    per={nm:{"m":0,"g":0,"dens":[]} for nm in test_names}
    for nm,t,v,gp in FRAMES:
        c=detect(v,sigma,foot,thr)
        ok=0
        if len(c) and len(gp):
            ok=int((cdist(gp*S,c*S).min(1)<=MATCH_UM).sum())
        per[nm]["m"]+=ok; per[nm]["g"]+=len(gp); per[nm]["dens"].append(len(c))
    return per
print("ok")

## 3 · v2 Sweep — gridi **daha çok tespit** yönüne uzat
v1'de recall doymamıştı. Küçük footprint + düşük eşik = daha çok tepe.

In [ ]:
SIGMAS=[(0.5,1,1), (1,2,2), (1,3,3)]
FOOTS =[(1,5,5), (3,7,7), (3,11,11), (5,15,15)]
THRS  =["otsu", "otsu*0.6", "p90"]

rows=[]; t0=time.time()
for sg,ft,th in itertools.product(SIGMAS,FOOTS,THRS):
    per=sweep_eval(sg,ft,th)
    M=sum(p["m"] for p in per.values()); G=sum(p["g"] for p in per.values())
    D=float(np.mean([np.mean(p["dens"]) for p in per.values()]))
    r=dict(sigma=str(sg),foot=str(ft),thr=th,
           recall=round(M/max(G,1),4), dens_ort=round(D,0))
    for nm in test_names:
        p=per[nm]
        r[nm[:9]]=round(p["m"]/max(p["g"],1),2)
        r[nm[:9]+"_d"]=int(np.mean(p["dens"]))
    rows.append(r)
    print(f"{str(sg):10s} {str(ft):11s} {th:9s} recall={r['recall']:.3f} dens={D:5.0f}")
res=pd.DataFrame(rows).sort_values("recall",ascending=False)
print(f"\n{len(rows)} konfig, {time.time()-t0:.0f}s\n")
print("=== EN IYI 12 (havuzlanmis recall) ===")
print(res.head(12).to_string(index=False))

### 3a · recall vs yoğunluk — dataset başına eğriler

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(15,5))
ax[0].scatter(res.dens_ort,res.recall,c="#4C78A8",s=60)
for r in res.head(3).itertuples():
    ax[0].annotate(f"{r.foot}\n{r.thr}",(r.dens_ort,r.recall),fontsize=7)
ax[0].set_xlabel("ortalama yoğunluk (tespit/kare)"); ax[0].set_ylabel("havuzlanmış recall")
ax[0].set_title("recall vs yoğunluk — doyuyor mu?")
for nm in test_names:
    ax[1].plot(res[nm[:9]+"_d"], res[nm[:9]], "o", label=nm, alpha=0.7)
ax[1].set_xscale("log"); ax[1].set_xlabel("o dataset'in yoğunluğu (log)"); ax[1].set_ylabel("recall")
ax[1].set_title("dataset başına recall vs yoğunluk"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG/"D11_sweep_v2.png",dpi=120,bbox_inches="tight"); plt.show()

## 4 · Kırık dataset teşhisi — kaçırılan GT'nin etrafında ne var?
`44b6_0b24845f`'te GT çekirdekleri yerel-maks olamıyor. Kaçırılanların çevresini görelim:
GT (kırmızı) + yakındaki tespitler (yeşil).

In [ ]:
BAD="44b6_0b24845f"
bs=eval(res.iloc[0].sigma); bf=eval(res.iloc[0].foot); bt=res.iloc[0].thr
print("konfig:",bs,bf,bt)
shown=0
fig,axes=plt.subplots(2,3,figsize=(15,10)); axes=axes.ravel()
for nm,t,v,gp in FRAMES:
    if nm!=BAD or shown>=6: continue
    c=detect(v,bs,bf,bt)
    D=cdist(gp*S,c*S) if len(c) else np.full((len(gp),1),1e9)
    for i,p in enumerate(gp):
        if shown>=6: break
        if D[i].min()<=MATCH_UM: continue          # sadece KACIRILANLAR
        z,y,x=int(round(p[0])),int(round(p[1])),int(round(p[2]))
        R=40
        y0,y1=max(0,y-R),min(v.shape[1],y+R); x0,x1=max(0,x-R),min(v.shape[2],x+R)
        a=axes[shown]; a.imshow(v[z,y0:y1,x0:x1],cmap="gray")
        a.plot(x-x0,y-y0,"rx",ms=14,mew=3,label="GT (kaçırıldı)")
        if len(c):
            near=c[(np.abs(c[:,0]-z)<=2)]
            for q in near:
                if y0<=q[1]<y1 and x0<=q[2]<x1:
                    a.plot(q[2]-x0,q[1]-y0,"g+",ms=10,mew=2)
        a.set_title(f"t={t} z={z} | en yakın tespit {D[i].min():.1f}µm\nGT parlaklık={v[z,y,x]:.0f}",fontsize=9)
        a.axis("off")
        if shown==0: a.legend(fontsize=8)
        shown+=1
for j in range(shown,6): axes[j].axis("off")
plt.suptitle(f"{BAD} — kaçırılan GT (kırmızı x) ve çevredeki tespitler (yeşil +)")
plt.tight_layout(); plt.savefig(FIG/"D12_missed_gt.png",dpi=120,bbox_inches="tight"); plt.show()

## 5 · **Gerçek** doğrulama — en iyi konfigleri tam tracking + metrikle ölç
Sweep sadece recall ölçüyor; asıl skor linking'i de içeriyor. En iyi 2 konfigi tam koşalım.
(~4.5 dk/konfig)

In [ ]:
def link_pairs(A,B):
    if len(A)==0 or len(B)==0: return []
    D=cdist(A*S,B*S); cost=np.where(D<=LINK_MAX_UM,D,1e6)
    r,c=linear_sum_assignment(cost)
    return [(int(i),int(j)) for i,j in zip(r,c) if D[i,j]<=LINK_MAX_UM]

def track_eval(nm, sigma, foot, thr):
    arr=open_image(TEST/(nm+".zarr")); T=arr.shape[0]
    cents=[detect(np.asarray(arr[t]).astype(np.float32),sigma,foot,thr) for t in range(T)]
    nodes=[]; edges=[]; off=[]; nid=1
    for t,c in enumerate(cents):
        off.append(nid)
        for p in c:
            nodes.append((nid,t,int(round(p[0])),int(round(p[1])),int(round(p[2])))); nid+=1
    for t in range(T-1):
        for i,j in link_pairs(cents[t],cents[t+1]):
            edges.append((off[t]+i,off[t+1]+j))
    gdf,ge=GT_ALL[nm]
    pn=pd.DataFrame(nodes,columns=["node_id","t","z","y","x"]); gmap={}
    for t,g in gdf.groupby("t"):
        p=pn[pn.t==int(t)]
        if len(p)==0 or len(g)==0: continue
        D=cdist(p[["z","y","x"]].values*S, g[["z","y","x"]].values*S)
        cost=np.where(D<=MATCH_UM,D,1e6); r,c=linear_sum_assignment(cost)
        pid=p["node_id"].values; gid=g["id"].values
        for i,j in zip(r,c):
            if D[i,j]<=MATCH_UM: gmap[int(pid[i])]=int(gid[j])
    gtset=set((int(u),int(v)) for u,v in ge); TP=0;FP=0;cov=set()
    for u,v in edges:
        gu=gmap.get(u); gv=gmap.get(v)
        if gu is None or gv is None: continue
        if (gu,gv) in gtset: TP+=1; cov.add((gu,gv))
        else: FP+=1
    FN=len(gtset)-len(cov)
    return dict(dataset=nm, jaccard=round(TP/max(TP+FP+FN,1),4),
                node_recall=round(len(set(gmap.values()))/max(len(gdf),1),4),
                TP=TP,FP=FP,FN=FN,pred_nodes=len(nodes))

RUN_FULL=True
if RUN_FULL:
    for k in range(2):
        cfg=res.iloc[k]; sg,ft,th=eval(cfg.sigma),eval(cfg.foot),cfg.thr
        print(f"\n=== KONFIG {k+1}: {sg} | {ft} | {th} (sweep recall={cfg.recall}) ===")
        out=[]; t0=time.time()
        for nm in test_names:
            r=track_eval(nm,sg,ft,th); out.append(r); print("  ",r)
        d=pd.DataFrame(out)
        print(f"  ORT jaccard={d.jaccard.mean():.4f} | ORT recall={d.node_recall.mean():.4f} "
              f"| ORT node/kare={d.pred_nodes.mean()/100:.0f} | {time.time()-t0:.0f}s")
    print("\n>> Referans (v1 baseline): ORT jaccard=0.6800, recall=0.7836")

## 6 · Karar

- [ ] En iyi konfig (gerçek metrikle) = **…**
- [ ] jaccard: 0.68 → **…**
- [ ] `44b6_0b24845f` düzeldi mi? Kaçırılan GT görselinde ne görüyoruz?
- [ ] Yoğunluk / fazla-tahmin cezası kabul edilebilir mi?

Kazanan konfigi `02_baseline.ipynb`'e taşı → tam koşu → submit.